In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV

In [2]:
df=pd.read_csv('cleaned_dataset.csv')
df['ageofhouse']=np.abs(df['Year Built']-df['Yr Sold'])
df.info()
df["TotalBath"] = (
    df["Full Bath"]
    + 0.5*df["Half Bath"]
    + df["Bsmt Full Bath"]
    + 0.5*df["Bsmt Half Bath"]
)

df["TotalPorchSF"] = (
    df["Open Porch SF"]
    + df["Enclosed Porch"]
    + df["3Ssn Porch"]
    + df["Screen Porch"]
)

df["TotalFinishedSF"] = (
    df["1st Flr SF"]
    + df["2nd Flr SF"]
    + df["BsmtFin SF 1"]
    + df["BsmtFin SF 2"]
)

df["HasGarage"] = (df["Garage Area"] > 0).astype(int)

df["HasBasement"] = (df["Total Bsmt SF"] > 0).astype(int)

df["YearsSinceRemodel"] = df["Yr Sold"] - df["Year Remod/Add"]

df["Remodeled"] = (
    df["Year Built"] != df["Year Remod/Add"]
).astype(int)
df.info()
df[['TotalBath','TotalPorchSF','TotalFinishedSF','HasGarage','HasBasement','YearsSinceRemodel','Remodeled']]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 81 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   MS SubClass      2930 non-null   int64  
 1   MS Zoning        2930 non-null   object 
 2   Lot Frontage     2440 non-null   float64
 3   Lot Area         2930 non-null   int64  
 4   Street           2930 non-null   object 
 5   Alley            198 non-null    object 
 6   Lot Shape        2930 non-null   object 
 7   Land Contour     2930 non-null   object 
 8   Utilities        2930 non-null   object 
 9   Lot Config       2930 non-null   object 
 10  Land Slope       2930 non-null   object 
 11  Neighborhood     2930 non-null   object 
 12  Condition 1      2930 non-null   object 
 13  Condition 2      2930 non-null   object 
 14  Bldg Type        2930 non-null   object 
 15  House Style      2930 non-null   object 
 16  Overall Qual     2930 non-null   int64  
 17  Overall Cond  

,TotalBath,TotalPorchSF,TotalFinishedSF,HasGarage,HasBasement,YearsSinceRemodel,Remodeled
0,2.0,62,2295.0,1,1,50,0
1,1.0,120,1508.0,1,1,49,0
2,1.5,36,2252.0,1,1,52,0
3,3.5,0,3175.0,1,1,42,0
4,2.5,34,2420.0,1,1,12,1
...,...,...,...,...,...,...,...
2925,2.0,0,1822.0,1,1,22,0
2926,2.0,0,1527.0,1,1,23,0
2927,1.5,32,1307.0,0,1,14,0
2928,2.0,38,2583.0,1,1,31,1


In [3]:
df=df.drop(columns=['Pool QC','Fence','Misc Feature'])
# df.drop(columns=['Alley'])

In [4]:
missing=df.isnull().mean()*100

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 85 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   MS SubClass        2930 non-null   int64  
 1   MS Zoning          2930 non-null   object 
 2   Lot Frontage       2440 non-null   float64
 3   Lot Area           2930 non-null   int64  
 4   Street             2930 non-null   object 
 5   Alley              198 non-null    object 
 6   Lot Shape          2930 non-null   object 
 7   Land Contour       2930 non-null   object 
 8   Utilities          2930 non-null   object 
 9   Lot Config         2930 non-null   object 
 10  Land Slope         2930 non-null   object 
 11  Neighborhood       2930 non-null   object 
 12  Condition 1        2930 non-null   object 
 13  Condition 2        2930 non-null   object 
 14  Bldg Type          2930 non-null   object 
 15  House Style        2930 non-null   object 
 16  Overall Qual       2930 

In [6]:
from sklearn.impute import SimpleImputer
import numpy as np

features = [
    'Mas Vnr Area',
    'Total Bsmt SF',
    '1st Flr SF',
    'Gr Liv Area',
    'Garage Yr Blt',
    'Garage Area'
]

imputer = SimpleImputer(strategy='median')
temp = imputer.fit_transform(df[features])

def iqr_range(data, multiplier=1.5):
    q1 = np.percentile(data, 25)
    q3 = np.percentile(data, 75)
    iqr = q3 - q1

    lower = q1 - multiplier * iqr
    upper = q3 + multiplier * iqr

    return lower, upper

li = []
for i in range(len(features)):
    li.append(iqr_range(temp[:, i]))

print(li)
temp.shape

[(-244.125, 406.875), (30.25, 2064.25), (114.625, 2145.625), (200.875, 2667.875), (1903.5, 2059.5), (-64.0, 960.0)]


(2930, 6)

In [7]:
for i, feature in enumerate(features):
    lower, upper = li[i]
    outliers = df[(df[feature] < lower) | (df[feature] > upper)]
    print(f"{feature}: {len(outliers)} outliers")

Mas Vnr Area: 203 outliers
Total Bsmt SF: 123 outliers
1st Flr SF: 43 outliers
Gr Liv Area: 75 outliers
Garage Yr Blt: 9 outliers
Garage Area: 42 outliers


In [8]:
lower, upper = li[i]
df['Mas Vnr Area'] = df['Mas Vnr Area'].clip(lower=lower, upper=upper)
df['Garage Yr Blt'] = df['Garage Yr Blt'].clip(*li[4])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 85 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   MS SubClass        2930 non-null   int64  
 1   MS Zoning          2930 non-null   object 
 2   Lot Frontage       2440 non-null   float64
 3   Lot Area           2930 non-null   int64  
 4   Street             2930 non-null   object 
 5   Alley              198 non-null    object 
 6   Lot Shape          2930 non-null   object 
 7   Land Contour       2930 non-null   object 
 8   Utilities          2930 non-null   object 
 9   Lot Config         2930 non-null   object 
 10  Land Slope         2930 non-null   object 
 11  Neighborhood       2930 non-null   object 
 12  Condition 1        2930 non-null   object 
 13  Condition 2        2930 non-null   object 
 14  Bldg Type          2930 non-null   object 
 15  House Style        2930 non-null   object 
 16  Overall Qual       2930 

In [9]:
X = df.drop("SalePrice", axis=1)
y = df["SalePrice"]

x_train, x_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [10]:
missing[(missing>0)]

Lot Frontage       16.723549
Alley              93.242321
Mas Vnr Type       60.580205
Mas Vnr Area        0.784983
Bsmt Qual           2.730375
Bsmt Cond           2.730375
Bsmt Exposure       2.832765
BsmtFin Type 1      2.730375
BsmtFin SF 1        0.034130
BsmtFin Type 2      2.764505
BsmtFin SF 2        0.034130
Bsmt Unf SF         0.034130
Total Bsmt SF       0.034130
Electrical          0.034130
Bsmt Full Bath      0.068259
Bsmt Half Bath      0.068259
Fireplace Qu       48.532423
Garage Type         5.358362
Garage Yr Blt       5.426621
Garage Finish       5.426621
Garage Cars         0.034130
Garage Area         0.034130
Garage Qual         5.426621
Garage Cond         5.426621
TotalBath           0.068259
TotalFinishedSF     0.034130
dtype: float64

In [11]:
num_missing = [
    'Lot Frontage',
    'Mas Vnr Area',
    'BsmtFin SF 1',
    'BsmtFin SF 2',
    'Bsmt Unf SF',
    'Total Bsmt SF',
    'Bsmt Full Bath',
    'Bsmt Half Bath',
    'Garage Yr Blt',
    'Garage Cars',
    'Garage Area',
    'TotalBath',
    'TotalFinishedSF'
]

ohe_missing = [
    'Mas Vnr Type',
    'Electrical',
    'Garage Type'
]

oe_missing = [
    'Bsmt Qual',
    'Bsmt Cond',
    'Bsmt Exposure',
    'BsmtFin Type 1',
    'BsmtFin Type 2',
    'Fireplace Qu',
    'Garage Finish',
    'Garage Qual',
    'Garage Cond'
]
num_no_missing = [
    'MS SubClass',
    'Lot Area',
    'Overall Qual',
    'Overall Cond',
    'Year Built',
    'Year Remod/Add',
    '1st Flr SF',
    '2nd Flr SF',
    'Low Qual Fin SF',
    'Gr Liv Area',
    'Full Bath',
    'Half Bath',
    'Bedroom AbvGr',
    'Kitchen AbvGr',
    'TotRms AbvGrd',
    'Fireplaces',
    'Wood Deck SF',
    'Open Porch SF',
    'Enclosed Porch',
    '3Ssn Porch',
    'Screen Porch',
    'Pool Area',
    'Misc Val',
    'Mo Sold',
    'Yr Sold',
    'ageofhouse',
    'TotalPorchSF'
]
ohe_no_missing = [
    'MS Zoning',
    'Street',
    'Land Contour',
    'Lot Config',
    'Neighborhood',
    'Condition 1',
    'Condition 2',
    'Bldg Type',
    'House Style',
    'Roof Style',
    'Roof Matl',
    'Exterior 1st',
    'Exterior 2nd',
    'Foundation',
    'Heating',
    'Central Air',
    'Sale Type',
    'Sale Condition',
    
]
oe_no_missing = [
    'Lot Shape',
    'Utilities',
    'Land Slope',
    'Exter Qual',
    'Exter Cond',
    'Heating QC',
    'Kitchen Qual',
    'Functional',
    'Paved Drive'
]

In [12]:
num_pipe=Pipeline([
    ('miss',SimpleImputer(strategy='median')),
    ('scl',StandardScaler())
])
num_pipe2 = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])
col_pipe = Pipeline([
    ('missing', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])
ohe_pipe = Pipeline([
    ('ohe', OneHotEncoder(
        drop='first',
        handle_unknown='ignore',
        sparse_output=False
    ))
])

In [13]:
col_pipe = Pipeline([
    ('missing', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])
ohe_pipe = Pipeline([
    ('ohe', OneHotEncoder(
        drop='first',
        handle_unknown='ignore',
        sparse_output=False
    ))
])
oe_pipe = Pipeline([
    ('missing', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('oe', OrdinalEncoder(
        categories=[
            # order for Bsmt Qual
            ['Missing', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
            # order for Bsmt Cond
            ['Missing', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
            # order for Bsmt Exposure
            ['Missing', 'No', 'Mn', 'Av', 'Gd'],
            # order for BsmtFin Type 1
            ['Missing', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
            # order for BsmtFin Type 2
            ['Missing', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
            # order for Fireplace Qu
            ['Missing', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
            # order for Garage Finish
            ['Missing', 'Unf', 'RFn', 'Fin'],
            # order for Garage Qual
            ['Missing', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
            # order for Garage Cond
            ['Missing', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
        ],
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])
oe_nomiss_pipe = Pipeline([
    ('oe', OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

In [14]:

ct = ColumnTransformer(
    transformers=[
        ('num_missing', num_pipe, num_missing),
        ('num_no_missing', num_pipe, num_no_missing),

        ('ohe_missing', col_pipe, ohe_missing),
        ('ohe_no_missing', ohe_pipe, ohe_no_missing),

        ('oe_missing', oe_pipe, oe_missing),
        ('oe_no_missing', oe_nomiss_pipe, oe_no_missing)
    ],
    remainder='drop'
)

In [15]:
model=LinearRegression()

In [16]:
pip=Pipeline([
    ('train',ct),
    ('fit',model)
])

In [17]:
pip.fit(x_train,y_train)

Pipeline(steps=[('train',
                 ColumnTransformer(transformers=[('num_missing',
                                                  Pipeline(steps=[('miss',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scl',
                                                                   StandardScaler())]),
                                                  ['Lot Frontage',
                                                   'Mas Vnr Area',
                                                   'BsmtFin SF 1',
                                                   'BsmtFin SF 2',
                                                   'Bsmt Unf SF',
                                                   'Total Bsmt SF',
                                                   'Bsmt Full Bath',
                                                   'Bsmt Half Bath',
                                                   'Garage Yr Blt',
                                                   'Garage Cars', 'Garage Area',
                                                   'TotalBath',
                                                   'TotalFinishedSF']),
                                                 ('num_no_mi...
                                                   'Bsmt Exposure',
                                                   'BsmtFin Type 1',
                                                   'BsmtFin Type 2',
                                                   'Fireplace Qu',
                                                   'Garage Finish',
                                                   'Garage Qual',
                                                   'Garage Cond']),
                                                 ('oe_no_missing',
                                                  Pipeline(steps=[('oe',
                                                                   OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                                  unknown_value=-1))]),
                                                  ['Lot Shape', 'Utilities',
                                                   'Land Slope', 'Exter Qual',
                                                   'Exter Cond', 'Heating QC',
                                                   'Kitchen Qual', 'Functional',
                                                   'Paved Drive'])])),
                ('fit', LinearRegression())])

In [18]:
pip.predict(x_test)

C:\Users\SURAJ\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\SURAJ\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


array([157365.91913387, 108955.27480271, 213151.16005263, 122235.03560141,
       121806.74022154, 199232.73071131, 158164.57336347, 145894.64149088,
        98024.44965984, 357629.03365591, 224688.09189292, 245393.90357739,
        60221.29575857, 120912.85807726,  41965.70673655, 177921.69821196,
       136766.37657629, 199235.33765102, 128206.78291497, 146440.7994642 ,
       172702.59643206, 133139.73965812, 188958.99571292, 197367.63565129,
       186137.72559622, 271376.55549081, 365888.2706096 , 231603.09684742,
       194025.01683967, 236336.37720702, 193672.88141242,  70553.25691484,
       222525.90965194,  89495.07787294, 123726.6883341 ,  53105.57820674,
       200561.1830272 , 322144.31057061, 181371.180917  , 289544.6628399 ,
       342045.58883013, 217589.34560423, 242294.12971216, 190014.8936247 ,
       121388.53868874, 332751.25872309, 184100.46916582, 149660.55407419,
       158348.20932535, 118643.687395  , 174477.11915097, 134422.91218262,
       143940.59186899, 1

In [19]:
y_pred = pip.predict(x_test)

print("R² Score :", r2_score(y_test, y_pred))
print("MAE      :", mean_absolute_error(y_test, y_pred))
print("MSE      :", mean_squared_error(y_test, y_pred))
print("RMSE     :", root_mean_squared_error(y_test, y_pred))

R² Score : 0.8877482884366683
MAE      : 17292.434433257447
MSE      : 899983477.7854444
RMSE     : 29999.724628493583


C:\Users\SURAJ\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\SURAJ\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [20]:
y_train_pred = pip.predict(x_train)

print("===== Training Metrics =====")
print("R²   :", r2_score(y_train, y_train_pred))
print("MAE  :", mean_absolute_error(y_train, y_train_pred))
print("MSE  :", mean_squared_error(y_train, y_train_pred))
print("RMSE :", root_mean_squared_error(y_train, y_train_pred))

===== Training Metrics =====
R²   : 0.9291919740925136
MAE  : 14462.284329216285
MSE  : 421006053.9090369
RMSE : 20518.432052889344


In [21]:
model2=DecisionTreeRegressor()

In [22]:
pip2=Pipeline([
    ('train1',ct),
    ('fit1',model2)
])
pip2.fit(x_train,y_train)

Pipeline(steps=[('train1',
                 ColumnTransformer(transformers=[('num_missing',
                                                  Pipeline(steps=[('miss',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scl',
                                                                   StandardScaler())]),
                                                  ['Lot Frontage',
                                                   'Mas Vnr Area',
                                                   'BsmtFin SF 1',
                                                   'BsmtFin SF 2',
                                                   'Bsmt Unf SF',
                                                   'Total Bsmt SF',
                                                   'Bsmt Full Bath',
                                                   'Bsmt Half Bath',
                                                   'Garage Yr Blt',
                                                   'Garage Cars', 'Garage Area',
                                                   'TotalBath',
                                                   'TotalFinishedSF']),
                                                 ('num_no_m...
                                                   'Bsmt Exposure',
                                                   'BsmtFin Type 1',
                                                   'BsmtFin Type 2',
                                                   'Fireplace Qu',
                                                   'Garage Finish',
                                                   'Garage Qual',
                                                   'Garage Cond']),
                                                 ('oe_no_missing',
                                                  Pipeline(steps=[('oe',
                                                                   OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                                  unknown_value=-1))]),
                                                  ['Lot Shape', 'Utilities',
                                                   'Land Slope', 'Exter Qual',
                                                   'Exter Cond', 'Heating QC',
                                                   'Kitchen Qual', 'Functional',
                                                   'Paved Drive'])])),
                ('fit1', DecisionTreeRegressor())])

In [23]:
y_pred2=pip2.predict(x_test)
print("R² Score :", r2_score(y_test, y_pred2))
print("MAE      :", mean_absolute_error(y_test, y_pred2))
print("MSE      :", mean_squared_error(y_test, y_pred2))
print("RMSE     :", root_mean_squared_error(y_test, y_pred2))

R² Score : 0.8436568655668235
MAE      : 22395.1023890785
MSE      : 1253488573.9863482
RMSE     : 35404.6405713481


C:\Users\SURAJ\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\SURAJ\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [24]:
y_train_pred2= pip.predict(x_train)

print("===== Training Metrics =====")
print("R²   :", r2_score(y_train, y_train_pred2))
print("MAE  :", mean_absolute_error(y_train, y_train_pred2))
print("MSE  :", mean_squared_error(y_train, y_train_pred2))
print("RMSE :", root_mean_squared_error(y_train, y_train_pred2))

===== Training Metrics =====
R²   : 0.9291919740925136
MAE  : 14462.284329216285
MSE  : 421006053.9090369
RMSE : 20518.432052889344


In [25]:
model3=RandomForestRegressor()

In [26]:
pip3=Pipeline([
    ('train2',ct),
    ('fit2',model3)
])
pip3.fit(x_train,y_train)

Pipeline(steps=[('train2',
                 ColumnTransformer(transformers=[('num_missing',
                                                  Pipeline(steps=[('miss',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scl',
                                                                   StandardScaler())]),
                                                  ['Lot Frontage',
                                                   'Mas Vnr Area',
                                                   'BsmtFin SF 1',
                                                   'BsmtFin SF 2',
                                                   'Bsmt Unf SF',
                                                   'Total Bsmt SF',
                                                   'Bsmt Full Bath',
                                                   'Bsmt Half Bath',
                                                   'Garage Yr Blt',
                                                   'Garage Cars', 'Garage Area',
                                                   'TotalBath',
                                                   'TotalFinishedSF']),
                                                 ('num_no_m...
                                                   'Bsmt Exposure',
                                                   'BsmtFin Type 1',
                                                   'BsmtFin Type 2',
                                                   'Fireplace Qu',
                                                   'Garage Finish',
                                                   'Garage Qual',
                                                   'Garage Cond']),
                                                 ('oe_no_missing',
                                                  Pipeline(steps=[('oe',
                                                                   OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                                  unknown_value=-1))]),
                                                  ['Lot Shape', 'Utilities',
                                                   'Land Slope', 'Exter Qual',
                                                   'Exter Cond', 'Heating QC',
                                                   'Kitchen Qual', 'Functional',
                                                   'Paved Drive'])])),
                ('fit2', RandomForestRegressor())])

In [27]:
y_train_pred3= pip3.predict(x_train)

print("===== Training Metrics =====")
print("R²   :", r2_score(y_train, y_train_pred3))
print("MAE  :", mean_absolute_error(y_train, y_train_pred3))
print("MSE  :", mean_squared_error(y_train, y_train_pred3))
print("RMSE :", root_mean_squared_error(y_train, y_train_pred3))

===== Training Metrics =====
R²   : 0.9837482389894946
MAE  : 5859.296646757679
MSE  : 96628732.18701364
RMSE : 9829.991464239103


In [28]:
y_pred3=pip3.predict(x_test)
print("R² Score :", r2_score(y_test, y_pred3))
print("MAE      :", mean_absolute_error(y_test, y_pred3))
print("MSE      :", mean_squared_error(y_test, y_pred3))
print("RMSE     :", root_mean_squared_error(y_test, y_pred3))

R² Score : 0.9088529368955218
MAE      : 15844.021109215018
MSE      : 730775947.2016279
RMSE     : 27032.86790560017


C:\Users\SURAJ\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\SURAJ\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [29]:
model4=GradientBoostingRegressor()


In [30]:
pip4=Pipeline([
    ('train3',ct),
    ('fit3',model4)
])


In [31]:
param_grid = {
    'fit3__n_estimators': [100, 200, 300],
    'fit3__learning_rate': [0.05, 0.1],
    'fit3__max_depth': [3, 4],
    'fit3__min_samples_split': [2, 5],
    'fit3__min_samples_leaf': [1, 2],
    'fit3__subsample': [0.8, 1.0]
}

In [32]:
grid = GridSearchCV(
    estimator=pip4,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,      # Uses all CPU cores
    verbose=2
)
grid.fit(x_train,y_train)

Fitting 5 folds for each of 96 candidates, totalling 480 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('train3',
                                        ColumnTransformer(transformers=[('num_missing',
                                                                         Pipeline(steps=[('miss',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scl',
                                                                                          StandardScaler())]),
                                                                         ['Lot '
                                                                          'Frontage',
                                                                          'Mas '
                                                                          'Vnr '
                                                                          'Area',
                                                                          'BsmtFin '
                                                                          'SF '
                                                                          '1',
                                                                          'BsmtFin '
                                                                          'SF '
                                                                          '2',
                                                                          'Bsmt '
                                                                          'Unf '
                                                                          'SF',
                                                                          'Total '
                                                                          'Bsmt '
                                                                          'SF',
                                                                          'Bsmt '
                                                                          'Full '
                                                                          'Bath',
                                                                          'Bsmt '
                                                                          'Half '
                                                                          'Bath',
                                                                          'Garage '
                                                                          'Yr '
                                                                          'Blt',
                                                                          'Garag...
                                                                          'Qual',
                                                                          'Exter '
                                                                          'Cond',
                                                                          'Heating '
                                                                          'QC',
                                                                          'Kitchen '
                                                                          'Qual',
                                                                          'Functional',
                                                                          'Paved '
                                                                          'Drive'])])),
                                       ('fit3', GradientBoostingRegressor())]),
             n_jobs=-1,
             param_grid={'fit3__learning_rate': [0.05, 0.1],
                         'fit3__max_depth': [3, 4],
                         'fit3__min_samples_leaf': [1, 2],
                         'fit3__min_samples_split': [2, 5],
                         'fit3

In [33]:
fn=grid.best_estimator_

In [34]:
grid.best_params_

{'fit3__learning_rate': 0.1,
 'fit3__max_depth': 3,
 'fit3__min_samples_leaf': 1,
 'fit3__min_samples_split': 5,
 'fit3__n_estimators': 300,
 'fit3__subsample': 1.0}

In [35]:
grid.best_score_

0.9191062266274915

In [38]:
y_fnpred=fn.predict(x_test)
r2 = r2_score(y_test, y_fnpred)
mae = mean_absolute_error(y_test, y_fnpred)
mse = mean_squared_error(y_test, y_fnpred)
rmse = np.sqrt(mse)

print(f"R² Score : {r2}")
print(f"MAE      : {mae}")
print(f"MSE      : {mse}")
print(f"RMSE     : {rmse}")

R² Score : 0.9301136271954539
MAE      : 13874.77314743808
MSE      : 560317343.6777371
RMSE     : 23671.023291732385


C:\Users\SURAJ\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\SURAJ\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
